# 🎵 Spotify Music Intelligence Platform
## Notebook 01 — Analyse exploratoire des données (EDA)

**Objectif** : comprendre la structure du dataset, identifier les distributions, détecter les anomalies et formuler des hypothèses avant la modélisation.

**Source** : Spotify Tracks Dataset — Kaggle (Maharshi Pandya, CC0)

> ⚠️ **Rappel épistémologique** : toutes les corrélations observées ici sont des associations statistiques. Elles ne prouvent pas de liens causaux. La popularité dépend de nombreux facteurs non capturés dans ce dataset (algorithmes de recommandation, marketing, contexte culturel).

---
## 0. Imports et configuration

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Style global
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid', palette='muted')

SPOTIFY_GREEN = '#1DB954'
ACCENT_COLOR  = '#7c3aed'

print(f'Python  : {sys.version.split()[0]}')
print(f'Pandas  : {pd.__version__}')
print(f'NumPy   : {np.__version__}')
print(f'Seaborn : {sns.__version__}')

---
## 1. Chargement et première inspection du dataset

**Question** : Quelle est la structure brute du dataset ? Combien de morceaux, de colonnes, de genres ?

In [ ]:
# Chargement du CSV brut
# Assurez-vous d'avoir placé dataset.csv dans data/raw/ avant d'exécuter
import sys
sys.path.insert(0, '..')  # pour importer src depuis notebooks/

from pathlib import Path
RAW_PATH = Path('../data/raw/dataset.csv')

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f'Dataset introuvable : {RAW_PATH}\n'
        'Téléchargez-le depuis Kaggle et placez-le dans data/raw/dataset.csv'
    )

df = pd.read_csv(RAW_PATH)

print(f'Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(f'\nColonnes disponibles :')
for col in df.columns:
    print(f'  • {col} ({df[col].dtype})')

In [ ]:
# Aperçu des premières lignes
print('Aperçu (5 premières lignes) :')
df.head()

In [ ]:
# Statistiques descriptives générales
print('Statistiques descriptives des variables numériques :')
df.describe().round(3)

---
## 2. Analyse des valeurs manquantes

**Question** : Y a-t-il des données incomplètes ? Dans quelles colonnes ? Quel pourcentage ?

In [ ]:
missing = pd.DataFrame({
    'count': df.isnull().sum(),
    'pct':   (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('pct', ascending=False)

missing_nonzero = missing[missing['count'] > 0]

if missing_nonzero.empty:
    print('✅ Aucune valeur manquante détectée dans le dataset.')
else:
    print(f'⚠️ {len(missing_nonzero)} colonne(s) ont des valeurs manquantes :')
    display(missing_nonzero)
    
    # Visualisation
    fig, ax = plt.subplots(figsize=(10, 4))
    missing_nonzero['pct'].plot(kind='barh', ax=ax, color=SPOTIFY_GREEN, alpha=0.85)
    ax.set_title('Pourcentage de valeurs manquantes par colonne', fontsize=13, fontweight='bold')
    ax.set_xlabel('% manquant')
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    plt.tight_layout()
    plt.savefig('../reports/figures/missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('\n→ Stratégie : médiane pour les colonnes numériques, "Unknown" pour les textes.')

---
## 3. Doublons

**Question** : Y a-t-il des lignes entièrement identiques ? Un même `track_id` peut-il apparaître plusieurs fois ?

In [ ]:
n_full_dups = df.duplicated().sum()
print(f'Lignes entièrement dupliquées : {n_full_dups:,}')

if 'track_id' in df.columns:
    id_counts = df['track_id'].value_counts()
    n_unique  = df['track_id'].nunique()
    n_total   = len(df)
    n_repeated = (id_counts > 1).sum()
    
    print(f'\ntrack_id uniques   : {n_unique:,} / {n_total:,} lignes')
    print(f'track_id répétés   : {n_repeated:,} (même morceau, genres différents)')
    print(f'Occurrences max    : {id_counts.max()}')
    print()
    print('→ Note : dans ce dataset, un même morceau peut appartenir à plusieurs genres.')
    print('  Ce n\'est PAS une erreur. Mais lors du split train/test, on utilisera')
    print('  GroupShuffleSplit par track_id pour éviter les fuites de données.')

---
## 4. Distribution de la popularité

**Question** : Comment est distribuée la popularité ? Est-elle symétrique ? Y a-t-il beaucoup de morceaux avec une popularité nulle ?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribution de la popularité', fontsize=14, fontweight='bold')

# Histogramme
axes[0].hist(df['popularity'], bins=50, color=SPOTIFY_GREEN, edgecolor='white', alpha=0.85)
axes[0].axvline(df['popularity'].mean(),   color='red',    linestyle='--', linewidth=2, label=f"Moyenne : {df['popularity'].mean():.1f}")
axes[0].axvline(df['popularity'].median(), color='orange', linestyle='--', linewidth=2, label=f"Médiane : {df['popularity'].median():.1f}")
axes[0].set_title('Histogramme')
axes[0].set_xlabel('Score de popularité (0–100)')
axes[0].set_ylabel('Nombre de morceaux')
axes[0].legend()

# Boxplot
axes[1].boxplot(df['popularity'].dropna(), vert=False,
                medianprops=dict(color='orange', linewidth=2),
                boxprops=dict(color=SPOTIFY_GREEN),
                whiskerprops=dict(color=SPOTIFY_GREEN),
                capprops=dict(color=SPOTIFY_GREEN),
                flierprops=dict(marker='o', color='red', alpha=0.3, markersize=3))
axes[1].set_title('Boxplot')
axes[1].set_xlabel('Score de popularité')
axes[1].set_yticks([])

plt.tight_layout()
plt.savefig('../reports/figures/popularity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistiques
pop_zero = (df['popularity'] == 0).sum()
print(f"\nStatistiques de popularité :")
print(df['popularity'].describe().round(2).to_string())
print(f"\nMorceaux avec popularité = 0 : {pop_zero:,} ({pop_zero/len(df)*100:.1f}%)")
print()
print("Observations :")
print("→ La distribution est asymétrique à droite (skewed right).")
print("→ Un grand nombre de morceaux ont une popularité nulle ou très faible.")
print("→ Cela reflète la réalité : la plupart des morceaux sont peu écoutés.")

---
## 5. Genres musicaux

**Question** : Quels genres sont les plus représentés ? Y a-t-il des déséquilibres ?

In [ ]:
if 'track_genre' in df.columns:
    # Nombre de morceaux uniques par genre
    genre_counts = df.groupby('track_genre')['track_id'].nunique().sort_values(ascending=False)
    
    print(f'Nombre de genres distincts : {df["track_genre"].nunique()}')
    print(f'\nTop 20 genres (morceaux uniques) :')
    
    fig, ax = plt.subplots(figsize=(12, 7))
    top_genres = genre_counts.head(20)
    colors = [SPOTIFY_GREEN if i < 5 else '#555577' for i in range(len(top_genres))]
    
    bars = ax.barh(top_genres.index[::-1], top_genres.values[::-1], color=colors[::-1], alpha=0.85)
    ax.set_title('Top 20 genres par nombre de morceaux uniques', fontsize=13, fontweight='bold')
    ax.set_xlabel('Nombre de morceaux uniques')
    
    # Ajouter les valeurs
    for bar, val in zip(bars, top_genres.values[::-1]):
        ax.text(val + 10, bar.get_y() + bar.get_height()/2, f'{val:,}',
                va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('../reports/figures/top_genres.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Popularité par genre
    genre_pop = df.groupby('track_genre')['popularity'].mean().sort_values(ascending=False).head(20)
    
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(genre_pop.index[::-1], genre_pop.values[::-1], color=ACCENT_COLOR, alpha=0.85)
    ax.set_title('Popularité moyenne par genre (Top 20)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Popularité moyenne')
    ax.axvline(df['popularity'].mean(), color='red', linestyle='--',
               label=f"Moyenne globale : {df['popularity'].mean():.1f}")
    ax.legend()
    plt.tight_layout()
    plt.savefig('../reports/figures/popularity_by_genre.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("Observations :")
    print("→ Certains genres sont sur-représentés (souvent car ce dataset encode les genres manuellement).")
    print("→ La popularité varie significativement selon le genre.")
    print("→ Limite : la popularité par genre peut être biaisée si certains genres ont peu de morceaux.")

---
## 6. Popularité vs. Énergie et Danceability

**Question** : Existe-t-il une relation entre la popularité et l'énergie ou la danceability d'un morceau ?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Popularité vs. caractéristiques audio', fontsize=14, fontweight='bold')

# Échantillon pour la lisibilité (10 000 points max)
df_sample = df.sample(min(10_000, len(df)), random_state=42)

for ax, feature, label in zip(
    axes,
    ['energy', 'danceability'],
    ['Énergie', 'Danceability']
):
    if feature not in df.columns:
        ax.text(0.5, 0.5, f'Colonne {feature} absente', ha='center', va='center')
        continue
    
    ax.scatter(df_sample[feature], df_sample['popularity'],
               alpha=0.15, color=SPOTIFY_GREEN, s=8)
    
    # Ligne de tendance (moyenne par décile)
    df_sample['_bin'] = pd.qcut(df_sample[feature], q=10, duplicates='drop')
    trend = df_sample.groupby('_bin', observed=True)['popularity'].mean()
    bin_centers = [interval.mid for interval in trend.index]
    ax.plot(bin_centers, trend.values, color='red', linewidth=2.5, label='Tendance (déciles)')
    
    r = df_sample[[feature, 'popularity']].corr().iloc[0, 1]
    ax.set_title(f'Popularité vs. {label}\n(corrélation de Pearson r = {r:.3f})', fontsize=11)
    ax.set_xlabel(label)
    ax.set_ylabel('Popularité')
    ax.legend()

plt.tight_layout()
plt.savefig('../reports/figures/popularity_vs_features.png', dpi=150, bbox_inches='tight')
plt.show()

print("Observations :")
print("→ La dispersion est très grande : pour une même énergie, la popularité varie de 0 à 100.")
print("→ La tendance (ligne rouge) peut montrer une légère association, mais elle est faible.")
print("→ Interprétation : l'énergie et la danceability seules ne prédisent pas la popularité.")
print("→ RAPPEL : corrélation ≠ causalité.")

---
## 7. Matrice de corrélation

**Question** : Quelles variables sont corrélées entre elles ? Y a-t-il des relations intéressantes à explorer ?

In [ ]:
numeric_cols = [
    'popularity', 'danceability', 'energy', 'loudness',
    'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo', 'duration_ms'
]
numeric_cols = [c for c in numeric_cols if c in df.columns]

corr = df[numeric_cols].corr().round(3)

fig, ax = plt.subplots(figsize=(12, 10))

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # masque le triangle supérieur

sns.heatmap(
    corr,
    ax=ax,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    cbar_kws={'label': 'Corrélation de Pearson'},
    # mask=mask  # décommentez pour n'afficher que le triangle inférieur
)

ax.set_title('Matrice de corrélation de Pearson', fontsize=14, fontweight='bold', pad=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../reports/figures/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Corrélations avec la popularité
pop_corr = corr['popularity'].drop('popularity').sort_values(key=abs, ascending=False)
print("\nCorrélations avec 'popularity' (triées par valeur absolue) :")
print(pop_corr.to_string())
print()
print("Observations :")
print("→ energy et loudness sont souvent corrélés entre eux (forte corrélation positive attendue).")
print("→ acousticness et energy sont souvent anti-corrélés.")
print("→ Les corrélations avec popularity sont généralement faibles.")
print("→ Interprétation : les caractéristiques audio ne capturent qu'une partie de la popularité.")

---
## 8. Distribution des caractéristiques audio

**Question** : Comment sont distribuées les variables audio ? Y a-t-il des valeurs aberrantes ?

In [ ]:
audio_features = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]
audio_features = [c for c in audio_features if c in df.columns]

n_cols = 3
n_rows = (len(audio_features) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for i, feature in enumerate(audio_features):
    ax = axes[i]
    ax.hist(df[feature].dropna(), bins=50, color=SPOTIFY_GREEN, edgecolor='none', alpha=0.85)
    ax.axvline(df[feature].mean(),   color='red',    linestyle='--', linewidth=1.5, alpha=0.8)
    ax.axvline(df[feature].median(), color='orange', linestyle='--', linewidth=1.5, alpha=0.8)
    ax.set_title(feature, fontweight='bold')
    ax.set_xlabel('Valeur')
    ax.set_ylabel('Fréquence')
    skew = df[feature].skew()
    ax.text(0.98, 0.95, f'Skew: {skew:.2f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9, color='gray')

# Masquer les sous-graphiques vides
for j in range(len(audio_features), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribution des caractéristiques audio\n(rouge = moyenne, orange = médiane)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/audio_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print("Observations clés :")
print("→ instrumentalness : très asymétrique (la majorité des morceaux ont des voix → valeur proche de 0).")
print("→ speechiness : asymétrique (la plupart des morceaux sont peu 'parlés').")
print("→ loudness : distribution relativement normale, centrée autour de -8 dB.")
print("→ tempo : distribution proche de la normale (60–200 BPM).")

---
## 9. Analyse de la durée, du tempo et de la valence

**Question** : Y a-t-il une relation entre la durée/tempo/valence et la popularité ?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Popularité vs. Durée, Tempo et Valence', fontsize=13, fontweight='bold')

for ax, (feature, label) in zip(axes, [
    ('duration_ms', 'Durée (ms)'),
    ('tempo',       'Tempo (BPM)'),
    ('valence',     'Valence (positivité)')
]):
    if feature not in df.columns:
        continue
    
    df_s = df.sample(min(8_000, len(df)), random_state=0)
    ax.scatter(df_s[feature], df_s['popularity'],
               alpha=0.1, color=SPOTIFY_GREEN, s=6)
    
    # Tendance
    try:
        df_s['_bin'] = pd.qcut(df_s[feature], q=10, duplicates='drop')
        trend = df_s.groupby('_bin', observed=True)['popularity'].mean()
        centers = [i.mid for i in trend.index]
        ax.plot(centers, trend.values, color='red', linewidth=2)
    except Exception:
        pass
    
    r = df_s[[feature, 'popularity']].corr().iloc[0, 1]
    ax.set_title(f'{label}\nr = {r:.3f}')
    ax.set_xlabel(label)
    ax.set_ylabel('Popularité')

plt.tight_layout()
plt.savefig('../reports/figures/duration_tempo_valence.png', dpi=150, bbox_inches='tight')
plt.show()

if 'duration_ms' in df.columns:
    df['duration_min'] = df['duration_ms'] / 60_000
    print(f"Durée médiane : {df['duration_min'].median():.1f} minutes")
    print(f"Durée max    : {df['duration_min'].max():.1f} minutes")
    print(f"Durée min    : {df['duration_min'].min():.1f} minutes")

print()
print("Observations :")
print("→ La durée a peu d'influence directe sur la popularité.")
print("→ Le tempo (BPM) n'est pas un prédicteur fort de popularité.")
print("→ La valence (positivité) montre une légère tendance selon les genres.")

---
## 10. Analyse par paires (Pairplot)

**Question** : Quelles paires de caractéristiques audio montrent des relations intéressantes ?

In [ ]:
# On prend un sous-ensemble pour la performance
pairplot_features = ['popularity', 'danceability', 'energy', 'valence', 'acousticness']
pairplot_features = [c for c in pairplot_features if c in df.columns]

df_pair = df[pairplot_features].sample(min(3_000, len(df)), random_state=42)

g = sns.pairplot(
    df_pair,
    diag_kind='kde',
    plot_kws=dict(alpha=0.15, color=SPOTIFY_GREEN, s=10),
    diag_kws=dict(color=SPOTIFY_GREEN, fill=True, alpha=0.5),
)
g.figure.suptitle('Pairplot des caractéristiques audio (3 000 échantillons)',
                   fontsize=13, fontweight='bold', y=1.02)

plt.savefig('../reports/figures/pairplot.png', dpi=120, bbox_inches='tight')
plt.show()

print("Observations :")
print("→ energy et acousticness : forte anti-corrélation attendue (les morceaux très énergiques")
print("  sont rarement acoustiques).")
print("→ danceability et valence : tendance positive (les morceaux dansants tendent à être plus joyeux).")
print("→ popularity : peu de tendances claires avec les features audio seules.")

---
## 11. Résumé et conclusions pour la modélisation

**Ce que l'EDA nous apprend :**

In [ ]:
print("=" * 60)
print("RÉSUMÉ DE L'ANALYSE EXPLORATOIRE")
print("=" * 60)

print(f"""
📊 Structure du dataset
   • {len(df):,} entrées × {df.shape[1]} colonnes
   • {df['track_id'].nunique():,} morceaux uniques
   • {df['artists'].nunique():,} artistes uniques  
   • {df['track_genre'].nunique() if 'track_genre' in df.columns else 'N/A'} genres

🎯 Variable cible (popularity)
   • Plage : {df['popularity'].min():.0f} – {df['popularity'].max():.0f}
   • Moyenne : {df['popularity'].mean():.1f} | Médiane : {df['popularity'].median():.1f}
   • Distribution très asymétrique (skewed right)
   • Beaucoup de morceaux avec popularité 0

⚠️  Points d'attention pour la modélisation
   1. Fuites de données : même track_id dans plusieurs genres
      → Utiliser GroupShuffleSplit par track_id
   2. Distribution asymétrique de la cible
      → Pas besoin de transformation pour ce projet
   3. Corrélations faibles feature ↔ popularité
      → R² attendu modéré (35-45%)
   4. Interprétation limitée
      → Les caractéristiques audio ne capturent pas
         les facteurs marketing et culturels

✅  Prochaines étapes
   → Phase 2 : Preprocessing (data/raw/ → data/processed/)
   → Phase 3 : Entraînement des modèles de régression
   → Phase 4 : Clustering K-Means
   → Phase 5 : Dashboard Streamlit
""")

---
*Notebook généré dans le cadre du projet **Spotify Music Intelligence Platform** — Portfolio Data Engineering*